# IC de Wilson y evaluación representativa en SIRAP

Reproduce el IC de Wilson para las proporciones del holdout ciego SAVH y establece reglas para evaluar el video sin inflar el tamaño muestral con cuadros u OCR repetidos. El intervalo no vuelve representativa una muestra parcial; la representatividad depende del marco muestral, las etiquetas completas y la unidad de análisis.

## Holdout OCR disponible

Se seleccionaron 58 imágenes al azar fuera de la partición de desarrollo; 57 tenían referencia humana aprobada. Se reportan conteos 42/57 (OCR base), 35/57 (OCR afinado) y cobertura 56/57. La imagen excluida se mantiene documentada. La inferencia queda limitada a imágenes elegibles SAVH y no a todo el desempeño operativo.

## Video de garita

La evaluación publicada contiene 30 lecturas con referencia, 21 sin referencia y dos pasadas dudosas excluidas. No calcular Wilson sobre las 30 lecturas: el ground truth es incompleto y varias lecturas pueden pertenecer a la misma pasada. El resultado 18/30 se mantiene descriptivo y condicionado a casos clasificables.

In [ ]:
from statistics import NormalDist

def wilson_interval(successes: int, trials: int, confidence_level: float = 0.95):
    """Wilson score CI. Input rows must be independent Bernoulli units."""
    if isinstance(successes, bool) or isinstance(trials, bool):
        raise TypeError("successes and trials must be integers")
    if not isinstance(successes, int) or not isinstance(trials, int):
        raise TypeError("successes and trials must be integers")
    if trials <= 0 or not 0 <= successes <= trials:
        raise ValueError("require trials > 0 and 0 <= successes <= trials")
    if not 0 < confidence_level < 1:
        raise ValueError("confidence_level must be between 0 and 1")
    p = successes / trials
    z = NormalDist().inv_cdf(0.5 + confidence_level / 2)
    z2 = z * z
    denom = 1 + z2 / trials
    center = (p + z2 / (2 * trials)) / denom
    half = z * (p * (1-p) / trials + z2 / (4 * trials**2))**0.5 / denom
    return {"successes": successes, "trials": trials, "estimate": p,
            "confidence_level": confidence_level,
            "lower": max(0.0, center-half), "upper": min(1.0, center+half),
            "method": "wilson_score"}


In [ ]:
counts = {
    "OCR base exact": (42, 57),
    "OCR afinado exact": (35, 57),
    "Cobertura OCR base": (56, 57),
    "Cobertura OCR afinado": (56, 57),
}
for name, (successes, trials) in counts.items():
    ci = wilson_interval(successes, trials)
    print(f"{name}: {successes}/{trials} = {ci['estimate']:.1%}; "
          f"IC Wilson 95% [{ci['lower']:.1%}, {ci['upper']:.1%}]")


## Diseño para corregir sesgos en la próxima evaluación

1. Definir población objetivo y periodo antes de muestrear.
2. Usar una unidad independiente por vehículo/pasada; no tratar cuadros ni lecturas repetidas como nuevos ensayos.
3. Crear estratos de garita/cámara, iluminación, distancia/tamaño de placa, ángulo/oclusión y vehículo según el alcance previsto. Sortear unidades dentro de cada estrato. Con asignación proporcional se puede presentar proporción conjunta y Wilson; con sobremuestreo, usar ponderación y un intervalo estratificado adecuado.
4. Hacer doble anotación humana ciega a las predicciones y adjudicar desacuerdos antes de puntuar.
5. No excluir fallos de detección ni casos difíciles. Completar etiquetas faltantes o informar tasa de no evaluabilidad y análisis de sensibilidad. Todo reemplazo se selecciona aleatoriamente del mismo marco.
6. Congelar conjunto, semilla, pesos, umbrales y criterio de legibilidad; no usar el holdout para ajustar el modelo.
7. Publicar conteos, denominador, exclusiones, método, intervalo, estratos y límites de generalización.

### Regla de reporte para el video actual

Hasta que las 51 lecturas y todas las pasadas del marco tengan referencia adjudicada, reportar 18/30 solo como EMR descriptivo de lecturas clasificables. Mantener 21/51 sin clasificar como no evaluables y las dos pasadas dudosas como pendientes; no imputarlas ni extrapolarlas. Un censo completo del video describe ese video; inferir a futuros vehículos requiere muestreo representativo de pasadas en varios periodos y condiciones.

In [ ]:
video_status = {
    "pipeline_reads": 51,
    "with_ground_truth": 30,
    "without_ground_truth": 21,
    "exact_among_classifiable": 18,
    "ambiguous_reference_passes": 2,
    "full_video_wilson": None,
    "reason": "incomplete ground truth and clustered reads; independent fully labelled units are required",
}
video_status
